# A crack crossing a plate: a short PhAST simulation

**Learning objective:** Import a notched-plate mesh, understand its supports and loading, run a dynamic phase-field calculation, and interpret the computed crack evolution and energy components.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CEMS-Lab/autumn-school/blob/main/notebooks/extensions/01_dynamic_plate_crossing.ipynb)

[Download notebook with exercises and worked solutions](https://cems-lab.github.io/autumn-school/notebooks/extensions/01_dynamic_plate_crossing.ipynb) · [Environment Setup](https://cems-lab.github.io/autumn-school/SETUP.md)

**Predict first:** Under vertical tension, in which direction will the crack grow from the horizontal notch? When a crack opens, how might the stored elastic energy change?

Prepared by Allamaprabhu Ani and Sathiskumar A. Ponnusami, CEMS-Lab, for the UKACM Autumn School 2026.

In [ ]:
#@title Set up the course environment { display-mode: "form" }
# Environment setup is measured separately from the classroom computation.
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys
import time

setup_started = time.perf_counter()
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("This PhAST snapshot supports Python 3.10–3.12. Select a compatible runtime; see Environment Setup.")

def locate_course_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "autumn-school", base / "teaching/ukacm_autumn_school_2026"):
            if (candidate / "notebooks/day2_helpers").is_dir() and (candidate / "vendor/PhAST/src").is_dir():
                return candidate.resolve()
    return None

COURSE_ROOT = locate_course_root()
if COURSE_ROOT is None and IN_COLAB:
    COURSE_ROOT = Path.cwd() / "autumn-school"
    if COURSE_ROOT.exists():
        raise RuntimeError("An incomplete autumn-school folder exists. Start a fresh runtime or select the complete course folder.")
    # A release tag or full commit can be supplied for a fixed course edition.
    course_ref = os.environ.get("PHAST_COURSE_REF", "main")
    subprocess.run(["git", "init", str(COURSE_ROOT)], check=True, capture_output=True)
    subprocess.run(["git", "-C", str(COURSE_ROOT), "remote", "add", "origin", "https://github.com/CEMS-Lab/autumn-school.git"], check=True)
    subprocess.run(["git", "-C", str(COURSE_ROOT), "fetch", "--depth", "1", "origin", course_ref], check=True, timeout=180)
    subprocess.run(["git", "-C", str(COURSE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True, capture_output=True)
if COURSE_ROOT is None:
    raise FileNotFoundError("Open the notebook from the complete course folder, including notebooks/, configs/ and vendor/. See Environment Setup.")

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", str(COURSE_ROOT / "vendor/PhAST"), "nbformat", "nbclient", "nbconvert"], check=True, timeout=600)

for package_dir in (COURSE_ROOT / "vendor/PhAST/src", COURSE_ROOT / "notebooks"):
    if str(package_dir) not in sys.path:
        sys.path.insert(0, str(package_dir))

revision = subprocess.run(["git", "-C", str(COURSE_ROOT), "rev-parse", "HEAD"], text=True, capture_output=True)
package_versions = {}
for package in ("torch", "numpy", "scipy", "matplotlib", "nbformat"):
    try:
        package_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        package_versions[package] = "install using Environment Setup"
import matplotlib.pyplot as plt
import numpy as np
import torch
from io import BytesIO
from IPython.display import Image, display
from day2_helpers import assets_dir

torch.set_default_dtype(torch.float64)
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 11, "axes.labelsize": 11, "axes.titlesize": 12,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
    "figure.titlesize": 13, "figure.dpi": 150, "axes.grid": True,
    "grid.alpha": 0.3, "grid.linestyle": "--", "lines.linewidth": 1.8,
})

def display_figure(figure, dpi=150, alt="Rendered teaching figure"):
    """Retain figures for reading in the book and in a fresh notebook session."""
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    display(Image(data=buffer.getvalue(), alt=alt))

# Include dependency imports and plotting configuration in the setup duration.
setup_summary = {
    "environment": "Google Colab" if IN_COLAB else "local",
    "python": sys.version.split()[0],
    "course_revision": revision.stdout.strip() if revision.returncode == 0 else "downloaded archive",
    "versions": package_versions,
    "setup_seconds": round(time.perf_counter() - setup_started, 3),
}
print(setup_summary)

computation_started=time.perf_counter()
torch.set_num_threads(1)

## 1. The specimen and physical model

The 40 mm square glass plate has a 20 mm geometric notch. The notch is already a cut in the imported mesh. The phase-field variable starts at $d=0$ and increases towards $d=1$ as a diffuse crack forms in the remaining ligament.

PhAST solves the discrete balance of momentum with an explicit central-difference update, then an implicit damage update:

$$\mathbf M\ddot{\mathbf u}+\mathbf f_{\mathrm{int}}(\mathbf u,d)=\mathbf f_{\mathrm{ext}},
\qquad d^{n+1}\in[d^n,1].$$

Here $\mathbf M$ is the mass matrix, $\mathbf u$ is displacement and $n$ indexes physical time. The spectral split separates tensile and compressive elastic energies. The AT2 energy is

$$\mathcal E(\mathbf u,d)=\int_\Omega
\left[g(d)\psi^+(\boldsymbol\varepsilon)+\psi^-(\boldsymbol\varepsilon)
+\frac{G_c}{2\ell}d^2+\frac{G_c\ell}{2}|\nabla d|^2\right]\,\mathrm d\Omega,$$

where $g(d)=(1-\eta)(1-d)^2+\eta$, $G_c$ is fracture toughness, $\ell$ is the regularisation length and $\eta$ is residual stiffness. The history of tensile energy supplies the irreversible damage update.

### Open the inputs

The portable YAML file contains the material, supports, loading and numerical settings. Its mesh is the checked public PhAST B3 example. Keeping that mesh fixed makes the calculation reproducible across computers.

In [ ]:
from day2_helpers.dynamic_practical import case_files, load_case, setup_figure
config_path, mesh_path = case_files()
config = load_case()
print("Mesh:", mesh_path.name)
print("Material:", config["material"])
print("Solver:", config["solver"])

### Inspect the mesh, supports and load

The left and right edges have $u_x=0$. The top and bottom move vertically by $+0.002\,s(t)$ mm and $-0.002\,s(t)$ mm. The smooth ramp $s(t)$ rises from zero to one over $20\,\mu$s, then remains constant until $100\,\mu$s. The 0.5 mm regularisation length determines the diffuse crack-band scale.

In [ ]:
fig = setup_figure()
display_figure(fig, alt="B3 imported triangular mesh, geometric notch, supports and smooth opening ramp")
plt.close(fig)

## 2. Run the plate calculation

Run one baseline calculation on the CPU. The public solver updates momentum and damage at each time step; the projected damage solve enforces $d^{n+1}\ge d^n$. Numerical checks accompany every update. The calculation exports sampled fields, energy components and a results table into its own output folder.

The helper starts the public PhAST command with the visible configuration. Expand its Python source to inspect the subprocess, residual checks and output routines. The solver subprocess has a 95-second safety cap; the complete uninterrupted practical is tested against a 120-second computation budget after environment setup.

In [ ]:
import tempfile
from day2_helpers.dynamic_practical import run_case, load_results
output_dir = Path(os.environ.get("PHAST_DYNAMIC_OUTPUT", tempfile.mkdtemp(prefix="phast-b3-"))) / "results"
output_dir = run_case(output_dir, timeout_seconds=95)
fields, summary = load_results(output_dir)
print({key: summary[key] for key in ["nodes", "elements", "steps", "sampled_states", "all_checks_pass"]})

## 3. Follow the computed crack

The images use the same damage colour scale throughout. Locate the initial notch, then follow the high-damage band into the remaining 20 mm ligament. The four fields below are selected from the saved calculation.

In [ ]:
from day2_helpers.dynamic_practical import result_figure
fig = result_figure(output_dir)
display_figure(fig, alt="Four actual B3 damage states showing initiation and propagation across the ligament")
plt.close(fig)

### Animate the result

Each frame below is a sampled solver state. Frames emphasise the first $40\,\mu$s, when the crack initiates and traverses the plate, and include the final $100\,\mu$s hold state. Playback speed is selected for viewing.

In [ ]:
from day2_helpers.dynamic_practical import make_animation
gif_path = make_animation(output_dir, frames=40)
display(Image(filename=str(gif_path), alt="Computed B3 crack propagation from notch to plate edge"))

## 4. Interpret the fields and energy

Stored elastic energy changes as the plate is loaded and the crack opens. Fracture energy measures the diffuse crack contribution, and kinetic energy describes motion. These are energies per unit specimen thickness. A complete energy balance additionally requires the work done at the moving boundaries.

In [ ]:
from day2_helpers.dynamic_practical import energy_figure
fig = energy_figure(output_dir)
display_figure(fig, alt="Stored elastic, fracture and kinetic energy versus physical time")
plt.close(fig)

### Read and export a results table

The CSV records the time, maximum damage, displacement and forward extent of nodes with $d\ge0.95$ within a 2 mm strip along the ligament. The extent is a threshold-dependent, mesh-resolved descriptor. The NPZ stores the mesh and sampled displacement/damage fields for further plotting; Zarr stores the public solver's trajectory outputs.

In [ ]:
import csv
from IPython.display import HTML
with (output_dir / "results.csv").open() as stream:
    rows = list(csv.DictReader(stream))
selected_rows = [rows[i] for i in np.linspace(0, len(rows)-1, 6, dtype=int)]
head = "<tr><th>Time [µs]</th><th>Maximum d</th><th>Extent [mm]</th></tr>"
body = "".join(f"<tr><td>{float(r['time_us']):.2f}</td><td>{float(r['maximum_damage']):.3f}</td><td>{float(r['extent_d095_mm']):.2f}</td></tr>" for r in selected_rows)
display(HTML("<table>" + head + body + "</table>"))
print("Saved files:", ", ".join(sorted(p.name for p in output_dir.iterdir() if p.is_file())))

## Key takeaways

- Geometry, material properties, supports and the loading schedule jointly define the fracture problem.
- The phase field resolves a crack as a continuous damage band whose width depends on $\ell$ and mesh resolution.
- The dynamic calculation couples explicit momentum updates with constrained implicit damage updates.
- Field snapshots, an animation, energy curves and numerical tables describe complementary aspects of the same simulation.

## Student exercises

### Exercise 1: resolve the time and length scales

How many numerical updates occur during the $20\,\mu$s ramp? Compare the regularisation length with the plate width. Explain why the smallest mesh element affects explicit computational cost.

<details><summary>Hint</summary>

Read `summary['dt_seconds']` and use $N_\mathrm{ramp}\simeq t_\mathrm{ramp}/\Delta t$. The explicit stability step scales with element size divided by elastic-wave speed.

</details>

<details><summary>Worked Solution</summary>

With $\Delta t\simeq1.6353\times10^{-8}$ s, the ramp spans approximately 1,223 updates. The regularisation length is $0.5/40=0.0125$ of the plate width. A smaller element reduces the stable time step and increases the update count for the same physical duration. Resolution along the entire crack path must also be considered when assessing a mesh.

</details>

### Exercise 2: measure a crack descriptor from saved fields

Use the existing saved fields to compare the forward extent obtained with thresholds $d\ge0.5$ and $d\ge0.95$. At an intermediate time, explain why the two values can differ. This exercise reuses the completed simulation.

<details><summary>Hint</summary>

Select nodes with $x>20$ mm and $|y-20|\le1$ mm, then apply each threshold to one row of `fields['damage']`. Subtract 20 mm from the largest selected x-coordinate.

</details>

<details><summary>Worked Solution</summary>

The lower threshold includes the partially damaged process zone ahead of the highly damaged band, so its extent can be greater. Both are discrete descriptors of a diffuse field. Report the threshold, strip and mesh when presenting either measure as a crack-length estimate.

</details>

## Continue exploring

The example is the lightweight public [PhAST B3 dynamic SENT case](https://github.com/CEMS-Lab/PhAST/tree/f6324f899f0701769810be117f27f1208f7a582e/examples/dynamic/B3_dynamic_sent), with the checked mesh and explicit classroom numerical settings recorded in the YAML. A separate fixed-mesh half-time-step calculation characterises temporal sensitivity. Spatial refinement and larger branching problems form further studies.

For a new geometry, create the mesh with Gmsh, identify boundary sets, then follow the same material → supports → loading → solve → inspect sequence. The current mesh lets this short practical focus on fracture and result interpretation.

### Computation record

The elapsed value below includes pauses when cells are run interactively. The course execution record measures uninterrupted computation separately from installation.

In [ ]:
elapsed_seconds = time.perf_counter() - computation_started
print({"elapsed_after_setup_seconds": round(elapsed_seconds, 2),
       "solver_subprocess_seconds": json.loads((output_dir / "runtime.json").read_text())["whole_process_seconds"],
       "environment": setup_summary["environment"]})